In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

print("Groq key set:",bool(os.environ["GROQ_API_KEY"]))

Groq key set: True


In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
# response = model.invoke("Say hello in malayalam")
# print(response.content)
print(model.invoke("Reply with just the word: ready").content)

ready


## Create a small sandbox folder for the Filesystem server

In [3]:
import pathlib

SANDBOX_DIR = pathlib.Path("mcp_sandbox").resolve()
SANDBOX_DIR.mkdir(exist_ok=True)

(SANDBOX_DIR / "notes.txt").write_text(
    "Project kickoff meeting is scheduled for next Monday at 10am.\n"
    "Attendees: Priya, Rahul, Aisha.\n"
)
print("Sandbox ready at:", SANDBOX_DIR)
print("Contents:", list(SANDBOX_DIR.iterdir()))

Sandbox ready at: /home/ronald/Desktop/python-pratice/ai-training/week-6/mcp_sandbox
Contents: [PosixPath('/home/ronald/Desktop/python-pratice/ai-training/week-6/mcp_sandbox/web_summary.txt'), PosixPath('/home/ronald/Desktop/python-pratice/ai-training/week-6/mcp_sandbox/notes.txt')]


## MCP Filesystem server to list available tools

In [13]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

filesystem_server_params = StdioServerParameters(
    command="mcp-server-filesystem",   # the installed binary, not "npx"
    args=[str(SANDBOX_DIR)],
)

async def list_filesystem_tools():
    async with stdio_client(filesystem_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Filesystem server exposes {len(tools)} tools:\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:80]}")

await list_filesystem_tools()

Filesystem server exposes 14 tools:

  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file ins
  - read_text_file: Read the complete contents of a file from the file system as text. Handles vario
  - read_media_file: Read a file and return it as a base64-encoded content block with its MIME type. 
  - read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than 
  - write_file: Create a new file or completely overwrite an existing file with new content. Use
  - edit_file: Make line-based edits to a text file. Each edit replaces exact line sequences wi
  - create_directory: Create a new directory or ensure a directory exists. Can create multiple nested 
  - list_directory: Get a detailed listing of all files and directories in a specified path. Results
  - list_directory_with_sizes: Get a detailed listing of all files and directories in a specified path, includi
  - directory_tree: Get a recursive tree view of 

## MCP Fetch server to list available tools

In [14]:
fetch_server_params = StdioServerParameters(
    command="python3",
    args=["-m", "mcp_server_fetch"],
)

async def list_fetch_tools():
    async with stdio_client(fetch_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Fetch server exposes {len(tools)} tool(s):\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:100]}")

await list_fetch_tools()

Fetch server exposes 1 tool(s):

  - fetch: Fetches a URL from the internet and optionally extracts its contents as markdown.

Although original


## MCP Git server to list available tools

In [6]:
import sys
import subprocess
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

def get_current_repo_root() -> str:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, check=True,
    )
    return result.stdout.strip()

repo_path = get_current_repo_root()
print("Detected repo root:", repo_path)

git_server_params = StdioServerParameters(
    command=sys.executable,          # <-- was "python3"
    args=["-m", "mcp_server_git", "--repository", repo_path],
)

async def list_git_tools():
    async with stdio_client(git_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print(f"Git server exposes {len(tools)} tool(s):\n")
            for t in tools:
                print(f"  - {t.name}: {(t.description or '')[:100]}")

await list_git_tools()

Detected repo root: /home/ronald/Desktop/python-pratice/ai-training
Git server exposes 12 tool(s):

  - git_status: Shows the working tree status
  - git_diff_unstaged: Shows changes in the working directory that are not yet staged
  - git_diff_staged: Shows changes that are staged for commit
  - git_diff: Shows differences between branches or commits
  - git_commit: Records changes to the repository
  - git_add: Adds file contents to the staging area
  - git_reset: Unstages all staged changes
  - git_log: Shows the commit logs
  - git_create_branch: Creates a new branch from an optional base branch
  - git_checkout: Switches branches
  - git_show: Shows the contents of a commit
  - git_branch: List Git branches


## ReAct agent with MCP tools

In [9]:
from langchain.agents import create_agent

async def run_filesystem_agent(question: str):
    async with stdio_client(filesystem_server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)

            agent = create_react_agent(model, tools)
            result = await agent.ainvoke({"messages": question})
            print(result["messages"][-1].content)

await run_filesystem_agent(
    "List the files in the allowed directory, then read notes.txt and tell me "
    "who the attendees of the meeting are.",
    # "Update attendees Rahul with John "
)

The attendees listed in **notes.txt** are:

- Priya  
- Rahul  
- Aisha


## Multi-server agent with MultiServerMCPClient

In [16]:
from langchain_mcp_adapters.client import MultiServerMCPClient

multi_client = MultiServerMCPClient({
    "filesystem": {
         "command":"mcp-server-filesystem",   # the installed binary, not "npx"
         "args":[str(SANDBOX_DIR)],
        "transport": "stdio",
    },
    "fetch": {
        "command": "python3",
        "args": ["-m", "mcp_server_fetch"],
        "transport": "stdio",
    },
})

all_tools = await multi_client.get_tools()
print("Combined tools:", [t.name for t in all_tools])

Combined tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories', 'fetch']


In [19]:
multi_agent = create_agent(model, all_tools)

result = await multi_agent.ainvoke({
    "messages": (
        "Fetch the content at https://www.lipsum.com/, write a two-sentence "
        "summary of it, and save that summary to a file called web_summary.txt "
        "in the allowed directory."
    )
})
print(result["messages"][-1].content)

The summary has been saved to **web_summary.txt** in the allowed directory.


In [20]:
# Verify the multi-server task actually happened, not just claimed to happen
summary_file = SANDBOX_DIR / "web_summary.txt"
if summary_file.exists():
    print(summary_file.read_text())
else:
    print("File not found -- check the agent's response above for what it actually did.")

Lorem Ipsum is placeholder dummy text originating from a 45 BC Latin work by Cicero, popularized in the 1960s for printing and later digital typesetting. It is widely used because its word distribution mimics natural English, helping designers focus on layout without distraction.
